# LM в интерактивной сессии Kaggle

Конфигурация прежняя: softmax / sparsemax / 1-ReLUmax, по 1000 шагов, одна модель с 1 слоем, embedding 768, 12 голов, persistent memory 2688, context 512. FP16, общий batch 512, warmup 25 / cosine decay 1000, validation каждые 100 шагов. Это сокращение авторского LM на 80000 шагов; один seed.


In [1]:
import sys
import subprocess
from pathlib import Path

assert Path('/kaggle/working').is_dir(), 'Подключите ядро Kaggle, а не локальный Python.'
REPO_REVISION = 'd60271526bc5521f2843235f661ec4e87bae11c3'
REPO_URL = 'https://github.com/kostkornilov/hw2.git'
BASE = Path('/tmp/hw2-runtime/lm')
REPO = BASE / 'repo'
BASE.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', REPO_REVISION], cwd=REPO, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() == REPO_REVISION
sys.path.insert(0, str(REPO / 'kaggle'))
import run_lm
assert Path(run_lm.__file__).resolve() == REPO / 'kaggle/run_lm.py'
print('Код:', REPO_REVISION)


Cloning into '/tmp/hw2-runtime/lm/repo'...


Код: d60271526bc5521f2843235f661ec4e87bae11c3


HEAD is now at d602715 Split ICL and sort into independent Kaggle jobs


## Проверка GPU и параметров

Печатаем параметры перед запуском. `MODE = "full"` означает прежние 1000 шагов на метод; `"smoke"` — четыре шага, если потребуется отдельная короткая проверка. Режим сам по себе обучение не запускает.


In [2]:
import json
import torch

MODE = 'full'
assert torch.cuda.device_count() == 2, 'Для этой конфигурации нужны две GPU (T4 ×2).'
print('GPU:', [torch.cuda.get_device_name(i) for i in range(2)])
config, limits = run_lm.configuration(REPO, MODE)
print('Методы:', config['methods'])
print('Шагов на метод:', limits['lm_steps'])
print(json.dumps(config['language_modeling'], indent=2))


GPU: ['Tesla T4', 'Tesla T4']
Методы: ['softmax', 'epanechnikov', 'relumax']
Шагов на метод: 1000
{
  "seed": 1337,
  "attention_backend": "adasplash",
  "dtype": "float16",
  "compile": true,
  "n_layer": 1,
  "n_embd": 768,
  "n_head": 12,
  "pmem_size": 2688,
  "block_size": 512,
  "batch_size": 8,
  "gradient_accumulation_steps": 32,
  "learning_rate": 0.005,
  "min_lr": 0.0001,
  "weight_decay": 0.1,
  "beta1": 0.9,
  "beta2": 0.95,
  "grad_clip": 1.0,
  "warmup_iters": 25,
  "lr_decay_iters": 1000,
  "decay_type": "cosine",
  "eval_iters": 10,
  "log_interval": 100
}


## Подготовка окружения

Создаём отдельное окружение Python 3.11 с проверенными зависимостями. Ядро ноутбука не переключается: обучение будет выполняться дочерними процессами в этом окружении. Обычно подготовка занимает несколько минут.


In [3]:
assert 'job' not in globals(), 'Job уже создан. Используйте существующий job; не запускайте эту ячейку повторно.'
job = run_lm.Job('lm', MODE, REPO, BASE)
print('Каталог результатов:', job.root)
try:
    job.environment()
except BaseException as error:
    job.finish(error)
    raise
print('Окружение готово:', job.python)


Каталог результатов: /kaggle/working/results/lm/full_20261010_150702_269387

=== lm: install_uv ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 72.7 MB/s eta 0:00:00

=== lm: python311 ===
Using CPython 3.11.17
Creating virtual environment with seed packages at: /tmp/hw2-runtime/lm/env
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source /tmp/hw2-runtime/lm/env/bin/activate

=== lm: dependencies ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 110.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 38.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 105.0 MB/s  0:00:00
   ━━━━━━━━━━━

## Данные, обучение трёх методов и экспорт

Эта ячейка сначала скачивает и токенизирует BabiStories, затем последовательно обучает три модели на двух GPU. Вывод содержит названия этапов и сообщения об обучении. В конце проверяются checkpoints, создаются таблицы, графики и архив.

Ячейку запускать **один раз**. При прерывании ошибки сохраняются вместе с имеющимися результатами; автоматического продолжения обучения здесь нет.


In [4]:
status = json.loads((job.root / 'job_status.json').read_text())
assert status['status'] == 'running', status
assert not (job.root / 'language_modeling').exists(), 'Обучение уже запускалось; повторный запуск в этот каталог запрещён.'
try:
    run_lm.lm(job)
except BaseException as error:
    job.finish(error)
    raise
else:
    job.finish()
print('Обучение завершено. Полный архив:', job.root.with_suffix('.zip'))



=== lm: extract_babistories ===

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:4 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 94371840 bytes (90 MiB)

Extracting archive: /tmp/hw2-runtime/lm/babistories/babistories-dataset.7z.001
--
Path = /tmp/hw2-runtime/lm/babistories/babistories-dataset.7z.001
Type = Split
Physical Size = 94371840
Volumes = 5
Total Physical Size = 440364747
----
Path = babistories-dataset.7z
Size = 440364747
--
Path = babistories-dataset.7z
Type = 7z
Physical Size = 440364747
Headers Size = 192
Method = LZMA2:24
Solid = +
Blocks = 1

Everything is Ok

Files: 2
Size:       1991103463
Compressed: 440364747

=== lm: tokenize ===

traindataset.txt: 0 stories [00:00, ? stories/s]
traindataset.txt: 560 stories [00:00, 5597.03 stories/s]
traindataset.txt: 1140 stories [00:00, 5714.28 stories/s]
traindataset.txt: 1722 stories [00:00, 5760.08 stories/s]
traindataset.txt: 2299 stories [00:00, 5642.63 st

: 

## Результаты и небольшой архив для скачивания

Просмотр готовой таблицы и графика. Отдельный небольшой ZIP содержит графики, таблицы, логи и метаданные; веса моделей входят только в полный архив. Скачайте архив через файловую панель Kaggle: `/kaggle/working/results/lm/`. Ячейка печатает точные пути и имя небольшого архива с суффиксом `_lite.zip`. Ссылки на файлы удалённого ядра не используются.


In [ ]:
from IPython.display import display, Markdown, Image
from zipfile import ZipFile, ZIP_DEFLATED

status = json.loads((job.root / 'job_status.json').read_text())
assert status['status'] == 'passed', status
result = job.root / 'language_modeling'
assert json.loads((result / 'artifact_validation.json').read_text())['status'] == 'passed'
display(Markdown((result / 'results_summary.md').read_text()))
display(Image(filename=str(result / 'loss_curves.png')))
light_archive = job.root.parent / (job.root.name + '_lite.zip')
with ZipFile(light_archive, 'w', compression=ZIP_DEFLATED) as archive:
    for path in sorted(job.root.rglob('*')):
        if path.is_file() and path.suffix in {'.png', '.pdf', '.csv', '.md', '.json', '.log', '.txt'}:
            archive.write(path, path.relative_to(job.root))
print('Лёгкий архив:', light_archive, f'({light_archive.stat().st_size / 1024**2:.2f} МБ)')
print('Полный архив с весами:', job.root.with_suffix('.zip'))
print('Скачайте архив через файловую панель Kaggle из указанного каталога /kaggle/working/results/lm/.')
